# Embed BAGEL metadata on a free Colab CPU runtime

This release notebook handles **only `bagel-7b-mot.safetensors`**, whose existing sidecar contains verified metadata. It does not quantize or load tensors. The server-side working copy is rewritten in place, and its tensor payload SHA-256 is checked before and after the header shift.

Add a private Colab secret named `HF_TOKEN` with write access to `6chan/bagel_comfy`. Keep `UPLOAD_TO_HUB = False` for the first run; inspect validation, then change it to `True` and run the upload cells.

In [ ]:
%pip install -q -U huggingface_hub hf_xet

import os
from pathlib import Path
from google.colab import userdata
from huggingface_hub import HfApi

os.environ["HF_XET_CHUNK_CACHE_SIZE_BYTES"] = "0"  # avoid a second large local cache
REPO_ID = "6chan/bagel_comfy"
FILENAME = "bagel-7b-mot.safetensors"
METADATA_FILENAME = FILENAME + ".comfyui-bagel.json"
CODE_BRANCH = "comfyui-native-migration"
WORK_DIR = Path("/content/bagel-metadata-release")
UPLOAD_TO_HUB = False

HF_TOKEN = userdata.get("HF_TOKEN")
assert HF_TOKEN, "Add HF_TOKEN in Colab Secrets before continuing"
identity = HfApi(token=HF_TOKEN).whoami()
print({"authenticated_as": identity["name"], "upload_enabled": UPLOAD_TO_HUB})

In [ ]:
import shutil
import subprocess

WORK_DIR.mkdir(parents=True, exist_ok=True)
free_bytes = shutil.disk_usage(WORK_DIR).free
model_info = HfApi(token=HF_TOKEN).model_info(REPO_ID, files_metadata=True)
remote_file = next(item for item in model_info.siblings if item.rfilename == FILENAME)
required_bytes = int(remote_file.size) + 5 * 1024**3
print({"free_gib": round(free_bytes / 1024**3, 1), "model_gib": round(remote_file.size / 1024**3, 1)})
assert free_bytes >= required_bytes, "This Colab runtime lacks enough disk; reconnect to a new CPU runtime"

code_dir = WORK_DIR / "ComfyUI-BAGEL"
if not code_dir.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", CODE_BRANCH, "https://github.com/neverbiasu/ComfyUI-BAGEL.git", str(code_dir)], check=True)

In [ ]:
from huggingface_hub import hf_hub_download

model_path = Path(hf_hub_download(REPO_ID, FILENAME, local_dir=WORK_DIR, token=HF_TOKEN))
metadata_path = Path(hf_hub_download(REPO_ID, METADATA_FILENAME, local_dir=WORK_DIR, token=HF_TOKEN))
print({"model": str(model_path), "metadata": str(metadata_path), "size_gib": round(model_path.stat().st_size / 1024**3, 1)})

In [ ]:
import subprocess

subprocess.run([
    "python",
    str(code_dir / "scripts/embed_bagel_metadata.py"),
    "--source", str(model_path),
    "--metadata", str(metadata_path),
    "--in-place",
], check=True)

In [ ]:
import json
import struct

with model_path.open("rb") as file:
    header_len = struct.unpack("<Q", file.read(8))[0]
    header = json.loads(file.read(header_len))
embedded = json.loads(header["__metadata__"]["comfyui_bagel"])
sidecar = json.loads(metadata_path.read_text())["metadata"]
assert embedded == sidecar
assert embedded["format"] == "comfyui_bagel"
assert embedded["format_version"] == 1
assert embedded["variant"] == "BAGEL-7B-MoT"
assert set(embedded["model_configs"]) == {"llm_config.json", "vit_config.json"}
print({"validated": True, "variant": embedded["variant"], "metadata_fields": sorted(embedded)})

In [ ]:
assert UPLOAD_TO_HUB, "Validation passed. Set UPLOAD_TO_HUB = True before publishing."
from huggingface_hub import CommitOperationAdd, CommitOperationDelete

commit = HfApi(token=HF_TOKEN).create_commit(
    repo_id=REPO_ID,
    repo_type="model",
    operations=[
        CommitOperationAdd(path_in_repo=FILENAME, path_or_fileobj=str(model_path)),
        CommitOperationDelete(path_in_repo=METADATA_FILENAME),
    ],
    commit_message="Embed ComfyUI-BAGEL metadata in bagel-7b-mot",
)
print({"commit_url": commit.commit_url, "commit_oid": commit.oid})

In [ ]:
import requests
from huggingface_hub import hf_hub_url

remote_url = hf_hub_url(REPO_ID, FILENAME, revision=commit.oid)
def fetch_remote_range(start, end):
    with requests.get(remote_url, headers={"Authorization": f"Bearer {HF_TOKEN}", "Range": f"bytes={start}-{end}"}, timeout=120, stream=True) as response:
        response.raise_for_status()
        assert response.status_code == 206, "Hub did not honor the Range request; refusing a full redownload"
        content = response.raw.read(end - start + 1)
        assert len(content) == end - start + 1, "Hub returned a truncated validation range"
        return content

remote_header_len = struct.unpack("<Q", fetch_remote_range(0, 7))[0]
assert remote_header_len <= 64 * 1024**2, "Remote safetensors header is implausibly large"
remote_header = json.loads(fetch_remote_range(8, 7 + remote_header_len))
remote_metadata = json.loads(remote_header["__metadata__"]["comfyui_bagel"])
assert remote_metadata == sidecar
print({"remote_header_validated": True, "revision": commit.oid})